# Q.4 Writing Viterbi Algorithm for the Primer
Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.


Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

### Hidden Markov Models (HMMs):

   It is a statistical model that represents systems with hidden states and observable events. It consists of:
   1. States: Hidden conditions that are not directly observable.
      - E: Exon regions (coding DNA)
      - I: Intron regions (non-coding DNA) 
      - 5: 5' splice site (transition region between exon and intron)
   2. Observations: Observable events infulenced by the hidden states.
   3. Transition Probabilities: Probabilities of moving from one state to another.
      - P(E→E): How likely we stay in an exon
      - P(E→5): How likely we transition from exon to splice site
      - P(5→I): How likely we go from splice site to intron
      - P(I→I): How likely we stay in an intron
      - etc.
   4. Emission Probabilities: Probabilities of observing a particular event given a state.
      - P(A|E): Probability of seeing an A in an exon
      - P(G|I): Probability of seeing a G in an intron
      - etc.
   5. Initial State Probabilties: Probabilities of the system starting in a particular state.



In [1]:
import numpy as np
import math

STATES = ['E', '5', 'I']
NUCLEOTIDES = ['A', 'C', 'G', 'T']

# 2. Initial Probabilities: Probability of starting with an E or an I
INIT_PROB = {'E': 1.0, '5': 0.0, 'I': 0.0}

# 3. Transition matrix: Probabilities of moving from one state to another
TRANS_PROB = {
    'Start': {'E':1.0, '5': 0, 'I': 0.0, 'End':0.0},
    'E': {'E': 0.9, '5': 0.1 , 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0 , 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0 , 'I': 0.9, 'End': 0.1}
}

# 4. Emission probabilities: Probability of emitting nucleotides (A, C, G, T) in each state
EMISSION_PROB = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}

def log(x):
    if(x == 0):
        return -math.inf
    else:
        return math.log(x)
    
# Log probability
def LOG_PROB(path, seq):
    log_prob = 0.0
    if len(path) != len(seq):
        raise ValueError("The length of state path and the observed sequence must be same")
    prev = 'Start'

    for i in range(len(seq)):
        curr = path[i]
        observe = seq[i]
        log_prob += log(TRANS_PROB[prev][curr])+log(EMISSION_PROB[curr][observe])
        prev = curr
        
    if(prev == 'I'):
        log_prob += log(TRANS_PROB[prev]['End'])

    return log_prob

STATE_PATH = "EEEEEEEEEEEEEEEEEE5IIIIIII"
OBSERVED_SEQ = "CTTCATGTGAAAGCAGACGTAAGTCA"
ans = LOG_PROB(STATE_PATH, OBSERVED_SEQ)

print(ans)

-41.21967768602254


In [2]:
def VITERBI(observed_sequence):
    num_states = len(STATES)
    num_obs = len(observed_sequence)
    VITERBI_MAT = np.full((num_states, num_obs), -np.inf)
    
    # For backtracking
    dp = np.zeros((num_states, num_obs), dtype=int)
    # Convert to state indices
    st_ind = {s: i for i, s in enumerate(STATES)}

    # Initializing the DP table
    for i, j in enumerate(STATES):
        VITERBI_MAT[i, 0] = log(INIT_PROB[j]) + \
        log(EMISSION_PROB[j][observed_sequence[0]])
        
    # Recursive part
    for k in range(1, num_obs):
        for curr_ind, curr_state in enumerate(STATES):
            max_log_prob = -np.inf
            best = 0
            
            for prev_ind, prev_state in enumerate(STATES):
                log_prob = VITERBI_MAT[prev_ind, k-1] + \
                log(TRANS_PROB[prev_state][curr_state])
                if log_prob > max_log_prob:
                    max_log_prob = log_prob
                    best = prev_ind
                    
            VITERBI_MAT[curr_ind, k] = max_log_prob + \
            log(EMISSION_PROB[curr_state][observed_sequence[k]])
            dp[curr_ind, k] = best

    ans = []
    st = np.argmax(VITERBI_MAT[:, -1])
    ans.append(STATES[st])

    for i in range(num_obs-1, 0, -1):
        st = dp[st, i]
        ans.insert(0, STATES[st])
    return ans, np.max(VITERBI_MAT[:, -1])

OBSERVERVED_SEQ = "CTTCATGTGAAAGCAGACGTAAGTCA"
ans, log_prob = VITERBI(OBSERVERVED_SEQ)
print(f"Most probable path {ans} ")
print(f"Log probability for path {log_prob}" )

Most probable path ['E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E'] 
Log probability for path -38.677666280562796


## The Viterbi Algorithm

1. **INITIALISATION**: Set up the first column of Viterbi matrix with initial probabilities
2. **RECURSION**: For each position in the sequence, calculate:
   - For each possible state, find the most likely previous state
   - Update our matrix with the new probability
   - Keep track of the dp (backpointers) to remember which path we took
3. **TERMINATE**: Find the most likely final state
4. **BACKTRACK**: Follow these backpointers to reconstruct the most likely state sequence


One interesting thing is that depending on our models parameters, the most likely path might actually just be all exons when transition probabilities heavily favor staying in one state.
Also I am amazed to see this algorithm is used all over the place like speech recognition , spell check , etc.
